# Text as Data Lab 0: Introduction, Python revision & Regular Expressions

The aims of the lab are to:
 - Introduce you to Colab, verify that you're set up with the correct python packages
 - Load textual data from Reddit as a JSON file
 - Explore the data to learn a bit about it
 - Review salient Python features, such as Counter and list comprehensions
 - Introduce regular expressions, a common tool for text matching and processing

**Before you start, save a copy of this lab to your drive using "File > Save a Copy in Drive".** If you skip this step, you may lose progress that you have made (e.g., if you close the browser tab or your computer crashes).

## Colab Introduction


Colab is a cloud-based Jupyter Notebook.  It is used internally by engineers and researchers at Google and companies worldwide to prototype and share data science and ML results in an easy-to-use way.

It supports:

1. Text Cells with [Markdown](https://github.com/adam-p/markdown-here/wiki/Markdown-Cheatsheet) formatting
2. Code Cells
3. Notebook stores code, output, and execution order
4. Tab and Tab + Tab Autocomplete
5. IPython Help Features
6. [IPython Magics (`%%`)](https://ipython.readthedocs.io/en/stable/interactive/magics.html) (such as [%%timeit](https://ipython.readthedocs.io/en/stable/interactive/magics.html#magic-timeit) to time the execution of a cell)

### Additional Features

- collaborative editing
- history
- comments
- executed code history
- Shift+click multiple cell selection
- searchable code snippets + table of contents
- scratchpad (⌘/Ctrl + Alt + N)

### Keyboard Shortcuts
| Command | Action |
| ---- | ----: |
|⌘/Ctrl+Enter | Run Selected Cell |
|Shift+Enter| Run Cell and Select Next |
|Alt+Enter| Run cell and insert new cell|
|⌘/Ctrl+M I | Interrupt Execution |

- You can open the command Palette to see all shortcuts by going to Tools --> Command palette.

### Summary of tips
- Use TAB to autocomplete an expression.
- You can also execute the code with a ? to get the documentation (e.g. `?sum`)
- In Jupyter / Colab you can execute shell commands using `!`, example: `!ls` to list the current files.

*Note:* Occasionally Colab may hang or crash (due to cloud flakiness or bad code).  You can control the execution using the Runtime menu to reboot and start fresh.  To resume where you left off you can click "Run before" and it will run all cells before the one currently selected.


## Downloading

Let's start by downloading a file containing data scraped from the online forum platform [Reddit](https://www.reddit.com/). This should take at most a few seconds to run.

In [ ]:
# Download data
!wget -O reddit_posts.json https://gla-my.sharepoint.com/:u:/g/personal/jake_lever_glasgow_ac_uk/EY_R8Y7DkrxMqXGe-zlgeNkBdJU5ZNTf8FYrN2pqDwddMA?download=1

## Data Exploration

If we take a look at the contents of the data file, we see that it is encoded as an array of objects in [JSON](https://en.wikipedia.org/wiki/JSON). There are a variety of other formats used for sharing text data (e.g., [XML](https://en.wikipedia.org/wiki/XML), [Protocol Buffers](https://en.wikipedia.org/wiki/Protocol_Buffers), etc.), but JSON is among the most common these days.

In [ ]:
# Look at the first 20 lines of the file - note the exclamation mark which tells Colab to run a terminal command instead of Python
!head -n 20 reddit_posts.json

Data provided by online APIs, such as the [Reddit API](https://www.reddit.com/dev/api/), are usually available in JSON. For this lab, we use a simplified version of the Reddit data, aggregating posts across several subreddits (i.e., sub-forums) and using just a handful of salient fields for each post. The data file consists of some metadata about the posts (`subreddit`, `score`, `id`, and `author`) and two fields with the text of the posts (`title` and `body`).

Now let's load the data into Python so we can work with it. The [json](https://docs.python.org/3/library/json.html) package in the Python standard library makes loading the data very easy.

In [ ]:
# Load the data into Python
import json
with open('reddit_posts.json', 'rt') as fin:
  reddit_posts = json.load(fin)

Python's JSON parser provides the data as a `list` of `dict` objects. You should already be familiar with these classes; if not, you can refer to [Python's data structures documentation](https://docs.python.org/3/tutorial/datastructures.html).

Let's look at the types of data available provided and some basic information.

In [ ]:
# Investigate the structure of the data
print('type(reddit_posts) =', type(reddit_posts))
print('len(reddit_posts) =', len(reddit_posts))
print('type(reddit_posts[0]) =', type(reddit_posts[0]))
print('reddit_posts[0].keys() =', reddit_posts[0].keys())
print('reddit_posts[0]["title"] =', reddit_posts[0]["title"])

We see that there are 2000 posts in the dataset. It's always important to understand the data you're working with, so let's see which subreddits these posts are from.

We could write code to loop through each of the posts and keep a running count of the number of posts in each subreddit:

```python
# The code that uses Counter below essentially does this:
subreddit_counts = {}
for post in reddit_posts:
  subreddit = post['subreddit']
  if subreddit not in subreddit_counts:
    subreddit_counts[subreddit] = 0
  subreddit_counts[subreddit] += 1
subreddit_counts
```

Python provides a convenient [`Counter`](https://docs.python.org/3/library/collections.html#collections.Counter) class to count values like this, so we'll use that instead. We can use a [Generator Expression](https://docs.python.org/3/reference/expressions.html#generator-expressions) to get the subreddit from each post and let `Counter` count how many times each subreddit appears.

In [ ]:
# Count the number of times each subreddit appears in this dataset
from collections import Counter
Counter(post['subreddit'] for post in reddit_posts)

We see that there are 9 subreddits covered by the dataset, mostly about beverages and gaming. The Soda subreddit is the least common (174 posts) and the NintendoSwitch subreddit is the most common (249 posts).

**Exercise:** Can you find the user with the most posts and how many posts they have? (Hint: `Counter` provides a function that will help you find the item with the highest count, without needing to look through all the counts manually. You can find this in the [documentation](https://docs.python.org/3/library/collections.html#collections.Counter).)

In [ ]:
# Exercise: Find the user with the most posts and how many posts they have
user_counter = Counter(post['author'] for post in reddit_posts)
most_common_user, num_posts = user_counter.most_common(1)[0]
print(f"User with most posts: {most_common_user} ({num_posts} posts)")

You should find that AutoModerator has the most number of posts, with 5.

AutoModerator sounds like it may not be a human user, so let's take a look at their posts. There are several ways to do this filtering. One of the most concise is to use a [List Comprehension](https://docs.python.org/3/tutorial/datastructures.html#list-comprehensions), so we'll do that here:

In [ ]:
[ post for post in reddit_posts if post['author'] == 'AutoModerator' ]

Indeed, through qualitatively inspecting of AutoModerator's posts, they appear to be automatically generated.

Depending on your application, you may want to filter out such posts. For instance, if you were trying to measure public sentiment about a product, you are probably only interested in human users. You could filter out auto-generated posts by manually checking each post, but this becomes impractical as the size of your collection grows. Later in this course we will cover text classification techniques, which could be used to automatically label a large number of posts given manually-labeled training data.



Can you find out anything else interesting from the dataset by inspecting the data?

In [ ]:
# Use this cell to explore the dataset. (Feel free to make other cells as well.)

## Regular Expressions

[Regular Expressions](https://en.wikipedia.org/wiki/Regular_expression) (often abbreviated as regex or regexp) are a common tool for pattern matching in text. Python provides regular expressions in the [`re`](https://docs.python.org/3/library/re.html) package.

Regular expressions can get very complicated, but it is valuable to be familiar with them. This portion of the lab provides a basic introduction to regex and provides links to further details if you want to learn more.

### Motivating Example

Let's say you want to find all posts that mention [Irn-Bru](https://en.wikipedia.org/wiki/Irn-Bru). One option would be to use the string contains operator (`in`):

In [ ]:
[ p for p in reddit_posts if 'Irn-Bru' in p['title'] or 'Irn-Bru' in p['body'] ]

Hmm, nothing matches? Didn't we see the first post mentioned Irn-Bru?

In [ ]:
reddit_posts[0]

Indeed so, but they stylized it as "Irn Bru" rather than "Irn-Bru". There's probably other ways people might write it too, like IrnBru, Irnbru, etc. We could come up with a list of possibilities and control for different casing (like the code below), but we still might miss some. There's a simpler way using regular expressions.

```python
match_strings = ['irn-bru', 'irn bru', 'irnbru']
[ p for p in reddit_posts if any(m in p['title'].lower() or m in p['body'].lower() for m in match_strings) ]
```

### Filtering with Regular Expressions

A regular expression defines a pattern to match in text as a string. Most characters (letters, numbers) in a pattern match themselves. Some perform special functions, like making the previous character optional or allowing multiple characters to match. For instances:
 - "`.`" matches any character.
 - "`?`" makes the previous character optional.

With this, we can define the regular expression `"irn.?bru"`, which will match `irn` and `bru` with any character (or no character) between them. An option allows case insensitive matching.

We can use regular expressions in python by first importing the `re` package and then compiling our pattern.

In [ ]:
import re
pattern = re.compile(r'irn.?bru', re.I) # re.I makes the pattern case insensitive

Note that the string with the regular expression has an `r` before. What's that about? That denotes that it is a raw string literal. It means that any [escape sequences](https://docs.python.org/3/reference/lexical_analysis.html#escape-sequences) (e.g. `\t` for a tab or `\\` for a backslash) are ignored. It's often a good idea to use raw string literals (i.e. putting an `r` before the string) for regular expressions. Regular expressions use a lot of backslashes, and it's annoying to have to escape them repeatedly (i.e. having to write `\\` instead of `\` every time).

The pattern object can do all sorts of things. Most commonly, you will use the `search` function, which finds and returns the first place that the pattern matches a string.

In [ ]:
match = pattern.search("Anyone tried Irn Bru?")
match

The match object gives the character offsets of the match (`match.span`) and the text itself that matches (`match.group(0)`).

If no match is found, `search` returns `None`, which evaluates as `False` when it's used in an `if` statement. This allows it to be easily used for filtering data. Using regular expressions, we find 6 posts about Irn-Bru.

In [ ]:
[ p for p in reddit_posts if pattern.search(p['title']) or pattern.search(p['body']) ]

We can see that there are a variety of ways that people express Irn-Bru.

**Exercise:** Can you write code that finds all the ways people express Irn-Bru in the dataset (e.g., "Irn-Bru", "IRN BRU", etc.)? Hint: you probably need to use the [`findall` function](https://docs.python.org/3/library/re.html#re.findall) with the `pattern` already defined above.

In [ ]:
# Exercise: Find all the ways people express Irn-Bru in the dataset
all_irn_bru_matches = []
for post in reddit_posts:
    all_irn_bru_matches += pattern.findall(post['title']) + pattern.findall(post['body'])

print(Counter(all_irn_bru_matches))

(You should find that "Irn Bru" appears 6 times, "IRN-BRU" appears twice, "IRN BRU" once, and "irn bru" once.)

Now let's try to find some years mentioned in the text of the documents. you'll need to define a new pattern. You might find that a [regular expressions cheatsheet](https://cheatography.com/davechild/cheat-sheets/regular-expressions/) is helpful so you can look up the pattern for `digit`.

**Exercise:** Find all 21st-century years (4 digits starting with '20') in the title and bodies of the posts. What is the most common?

In [ ]:
# Exercise: Find all 21st-century years (4 digits starting with '20') and the most common one
year_pattern = re.compile(r'20\d{2}')

all_years = []
for post in reddit_posts:
    all_years += year_pattern.findall(post['title']) + year_pattern.findall(post['body'])

year_counter = Counter(all_years)
print("All year counts:", year_counter)
print("\nMost common year:", year_counter.most_common(1)[0])

You should find that the digits `2016` appears 35 times across the posts.

Now, a question: do you think all of those 4 digit numbers are actually years in the text. Could they be anything else?

### Additional Resources

These examples only scratch the surface of the capacities of regular expressions. We refer you to these resources for further details.

 - [RegExr](https://regexr.com/) -- Interactively build and evaluate regular expressions in a GUI. This can be helpful when trying to build a tricky regex or figure out why it's not matching what you want it to.
 - [RegexOne](https://regexone.com/) -- A detailed and interactive regular expression tutorial
 - [Python's `re` documentation](https://docs.python.org/3/library/re.html) -- Provides details about both regex syntax and Python's regex API

## End

This is the end of Lab 0. Let us know if you encounter any issues! Remember we are here to help!

In this lab, you...
- started using Colab
- loaded textual data from Reddit as a JSON file
- explored the data to learn a bit about it
- reviewed salient Python features, such as Counter and list comprehensions
- explored using regular expressions in Python

**Please remember to submit your completed lab and feedback form on Moodle.**


## Optional Extras

The [cheatsheet](https://cheatography.com/davechild/cheat-sheets/regular-expressions/) may be handy for some of these. See if you can come up with a single regular expression for each problem.

- Count how many Reddit posts in our dataset have titles that start with a digit followed by any character that isn't a digit (e.g. "1A" or "7-").
- Count how many Reddit posts in our dataset have two 4-digit numbers in the body of the post? What about 4-digit numbers that are not part of larger bits of text (e.g. are properly separated - you could use `\b` to indicate a [word boundary](https://www.regular-expressions.info/wordboundaries.html)).
- Find Reddit posts where the title starts and ends with the same character. You likely need to use [backreferences](https://www.regular-expressions.info/backref.html) in the regular expression.
- Write a regex that matches to [British postcodes](https://ideal-postcodes.co.uk/guides/uk-postcode-format) (e.g. G12 8RZ) and is agnostic if there is a space in the middle. You can make it fairly general or try to be more specific to the intricacies of the postcode standards. See if there are any matches in the Reddit posts. Are they actually postcodes?
- Write a regex using the [re.sub function](https://docs.python.org/3/library/re.html#re.sub) that swaps the first part (the outcode) and second part (the incode) of a postcode (G128RZ -> 8RZG12). You could use backreferences as mentioned in the [re.sub documentation](https://docs.python.org/3/library/re.html#re.sub).